In [1]:
import os
import importlib

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  IN_COLAB = True
except:
  IN_COLAB = False

In [3]:
# if IN_COLAB:
#   %cd /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/SliceGPTModifications/
# else:
#   %cd ../SliceGPTModifications

# try:
#   import slicegpt
# except:
#   !pip install -e .
#   import slicegpt

In [4]:
# if IN_COLAB:
#   %cd /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/SliceGPTModifications/
# else:
#   %cd ../

# try:
#   import sliced_rag
# except:
#   !pip install -e .
#   import sliced_rag

In [5]:
if IN_COLAB:
    # !pip install -q "peft==0.13.2" "transformers==4.50.0" "lm_eval==0.4.1" "datasets==2.18.0"
    !pip install -q "peft==0.13.2" "transformers==4.57.3" "lm_eval==0.4.9.2" "datasets==2.18.0"
    !pip install -q "lm_eval[hf]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 137.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.7 MB/s eta 0:00:00


In [7]:
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

if IN_COLAB:
    os.makedirs(LOG_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(BASE_EVAL_DIR, exist_ok=True)

In [8]:
# import lm_eval
# importlib.reload(slicegpt)
# importlib.reload(sliced_rag)
# importlib.reload(lm_eval)

In [9]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

In [10]:
# model_id = "google/gemma-3-270m-it"
# model_id = "google/gemma-3-1b-it"

In [11]:
# def load_model(model_id):
#   model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
#   tokenizer = AutoTokenizer.from_pretrained(model_id)
#   return model, tokenizer

In [12]:
# model, tokenizer = load_model(model_id)

In [13]:
# def generate_text(prompt):
#   model_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#   generated_ids = model.generate(**model_inputs, max_new_tokens=50, do_sample=False)
#   generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
#   return generated_text

In [14]:
# prompt = """
# Question: What is the capital of France?

# Answer: """

In [15]:
# print(generate_text(prompt))

In [16]:
# from sliced_rag.evaluation import eval_baseline


In [37]:
model = "google/gemma-3-270m-it"
# model = "google/gemma-3-1b-it"
model_name = model.replace("/", "-")
# sparsities = [0.0, 0.10, 0.25, 0.40, 0.60]

# Evaluation

Running Gemma3 eval separately because of differences in the HF transformers library versions (Gemma3 requires transformers>=4.50, while slicegpt (and therefore sliced_rag) requires transformers==4.41.0)

In [45]:
import json
import logging
import sys
# import os

import lm_eval
from lm_eval import tasks
from lm_eval import utils as lm_eval_utils
from lm_eval.api.registry import ALL_TASKS
from lm_eval.models.huggingface import HFLM
# from lm_eval.tasks import initialize_tasks

# from slicegpt import gpu_utils, hf_utils, utils
# from slicegpt.config import config

# import os
# import json
# import torch
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import evaluate
# from tqdm import tqdm

# initialize_tasks()

logging.basicConfig(
    level=logging.DEBUG,
    stream=sys.stdout,
    format="%(levelname)s: %(message)s"
)

def get_logger():
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)

    # remove handlers if already set, to avoid double logging
    for h in list(logger.handlers):
        logger.removeHandler(h)

    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)

    formatter = logging.Formatter('%(levelname)s - %(message)s')
    handler.setFormatter(formatter)

    logger.addHandler(handler)

    return logger

def eval_baseline(args):
    """
    Run LM Evaluation Harness on either:
      - a sliced (pruned) model, if 'sliced_model_path' is provided
      - or a dense HF model, if not.
    """
    logger = get_logger()
    logger.info("Running Evaluation")

    # ---------- DENSE BASELINE BRANCH ----------
    logger.info(
        f"Loading DENSE baseline model {args['model']} directly from HF hub"
    )

    # Here we let LM Eval Harness load the model & tokenizer itself
    hflm = HFLM(
        pretrained=args["model"],
        batch_size=args["batch_size"],
        dtype="float16"
    )
    hflm.model.config.use_cache=False

    # ---------- Task selection ----------
    if args["tasks"] is None:
        logger.warning(
            "args['tasks'] is None -> using ALL_TASKS. "
            "This may be very slow / memory-heavy."
        )
        task_names = tasks.ALL_TASKS
    else:
        task_names = args["tasks"]

    logger.info(f"Selected Tasks: {task_names}")

    # ---------- Run evaluation ----------
    results = lm_eval.simple_evaluate(
        hflm,
        tasks=task_names,
        num_fewshot=args["num_fewshot"],
        batch_size=args["batch_size"],
        limit=args["limit"],
        write_out=True,
        log_samples=args["log_samples"]
    )

    logger.info("Results (metrics only):")
    # logger.info(results["results"])
    logger.info(results)


    # ---------- Save results ----------
    os.makedirs(args["save_dir"], exist_ok=True)

    # Use sparsity=0.0 if not present, so filenames still make sense
    sparsity_val = float(args.get("sparsity", 0.0))
    sparsity_tag = f"{sparsity_val:.2f}"

    result_path = os.path.join(
        args["save_dir"],
        f"results_s{sparsity_tag}_{'_'.join(task_names)}_light.json",
    )

    with open(result_path, "w") as f:
        try:
          json.dump(results, f, indent=2, skipkeys=True)
        except:
          json.dump(results["results"], f, indent=2, skipkeys=True)

    logger.info(f"Saved results to {result_path}")

    return results


In [46]:
EVAL_DIR = os.path.join(BASE_EVAL_DIR, f"{model_name}_unpruned")

args = {
    "model": model,
    "sliced_model_path": None,
    "sparsity": 0.0,
    "save_dir": EVAL_DIR,
    "tasks": ["squadv2"],
    "num_fewshot": 0,
    "batch_size": 32,
    "round_interval": 8,
    "limit": None,
    "log_samples": False
}

results = eval_baseline(args)

INFO - Running Evaluation
INFO - Loading DENSE baseline model google/gemma-3-270m-it directly from HF hub
INFO - Using device 'cuda'
INFO - Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}
INFO - Selected Tasks: ['squadv2']
INFO - Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
INFO - Using pre-initialized model
WARNING - None: No `generation_kwargs` specified in task config, defaulting to {'until': ['\n\n'], 'do_sample': False, 'temperature': 0}
INFO - Selected tasks:
INFO - Task: squadv2 (squadv2/squadv2.yaml)
INFO - squadv2: Using gen_kwargs: {'until': ['\n\n'], 'do_sample': False, 'temperature': 0}
WARNING - Overwriting default num_fewshot of squadv2 from None to 0
INFO - Building contexts for squadv2 on rank 0...


100%|██████████| 11873/11873 [00:00<00:00, 133299.17it/s]

INFO - Task: ConfigurableTask(task_name=squadv2,output_type=generate_until,num_fewshot=0,num_samples=11873); document 0; context prompt (starting on next line):    
Title: Normans

Background: The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually merge with the Carolingian-based cultures of West Francia. The distinct cultural and ethnic identity of the Normans emerged initially in the first half of the 10th century, and it continued to evolve over the succeeding centuries.

Question: In what country is Normandy located?

Answer:
(end of


Running generate_until requests: 100%|██████████| 11873/11873 [1:08:00<00:00,  2.91it/s]

INFO - Running loglikelihood requests



Running loglikelihood requests:   0%|          | 0/11873 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 12.55 GiB. GPU 0 has a total capacity of 22.16 GiB of which 8.34 GiB is free. Process 30675 has 13.82 GiB memory in use. Of the allocated memory 13.56 GiB is allocated by PyTorch, and 28.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
results